# Sub-basin aggregation of **all** variables (ET₀ output → Excel)

Replaces the two-variable version: this reads the NetCDFs written by `weathernext3_eto.py`
(`*_hourly.nc` and `*_daily.nc`) and aggregates **every** variable they contain — ET₀,
precipitation, temperature, dewpoint, VPD, 2 m wind, net and solar radiation, pressure,
elevation — to each sub-basin.

Masking follows the same convention as before: the shapefile is reprojected to EPSG:4326, cell
centres inside each polygon are selected with `shapely.contains_xy`, and the spatial average is
`cos(latitude)`-weighted. A basin smaller than one grid cell falls back to the cells it touches.

One workbook per input file, `<stem>_subbasins.xlsx`:

* `Summary` — one row per sub-basin: cell count, basin elevation, totals and peaks for ET₀ and
  precipitation, mean/min/max for every other variable
* `Variables` — data dictionary: units, description, how each variable was aggregated
* one sheet per variable — rows = valid time (with `Lead hour`), columns = sub-basins

In [ ]:
#@title Sub-basin aggregation of ALL variables — settings
!pip install -q geopandas openpyxl

SHP_PATH  = "sub_basins.shp"  #@param {type:"string"}   # upload .shp/.shx/.dbf/.prj together
NAME_COL  = ""                       #@param {type:"string"}   # "" = auto-detect
NC_GLOBS  = ["weathernext3_eto_*_hourly.nc", "weathernext3_eto_*_daily.nc"]
# One workbook per NetCDF: <file stem>_subbasins.xlsx

In [ ]:
import glob
import os
import re

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
import xarray as xr

nc_files = sorted({f for pat in NC_GLOBS for f in glob.glob(pat)})
assert nc_files, f"No NetCDF files matched {NC_GLOBS}"

gdf = gpd.read_file(SHP_PATH).to_crs(4326)
if not NAME_COL:
    cand = [c for c in gdf.columns if c != "geometry" and not pd.api.types.is_numeric_dtype(gdf[c])]
    NAME_COL = cand[0] if cand else None
names = gdf[NAME_COL].astype(str).tolist() if NAME_COL else [f"basin_{i+1}" for i in range(len(gdf))]
print(f"{len(gdf)} sub-basins (name column: {NAME_COL})")
print("files:", [os.path.basename(f) for f in nc_files])

In [ ]:
# Variables accumulated over the step (basin TOTAL in the summary); everything else is a mean.
SUM_LIKE = ("eto", "precipitation")

def basin_means(ds):
    """Area-weighted (cos latitude) basin means for every variable. Returns dict name -> Dataset."""
    flat = ds.stack(cell=("latitude", "longitude"))
    lon2d, lat2d = np.meshgrid(ds["longitude"].values, ds["latitude"].values)
    dx = float(abs(np.diff(ds["longitude"].values)).max())
    dy = float(abs(np.diff(ds["latitude"].values)).max())
    half_diag = 0.5 * float(np.hypot(dx, dy))

    out, cells = {}, {}
    for name, geom in zip(names, gdf.geometry):
        inside = shapely.contains_xy(geom, lon2d, lat2d)
        n = int(inside.sum())
        if n == 0:                                   # basin smaller than one grid cell
            inside = shapely.contains_xy(geom.buffer(half_diag), lon2d, lat2d)
            print(f"  {name}: smaller than a grid cell — {int(inside.sum())} touched cell(s) used")
        sub = flat.isel(cell=np.flatnonzero(inside.ravel()))
        w = np.cos(np.deg2rad(sub["latitude"]))
        out[name] = sub.weighted(w).mean("cell")
        cells[name] = n
    return out, cells

def sheet_name(var, used):
    s = re.sub(r"[\[\]:*?/\\]", "_", var)[:31]
    while s in used:                                  # Excel sheet names must be unique
        s = s[:28] + f"_{len(used)}"
    used.add(s)
    return s

In [ ]:
written = []
for path in nc_files:
    ds = xr.open_dataset(path)
    series, n_cells = basin_means(ds)
    stem = os.path.splitext(os.path.basename(path))[0]
    xlsx = f"{stem}_subbasins.xlsx"

    time_vars   = [v for v in ds.data_vars if "time" in ds[v].dims]
    spatial     = {v for v in ds.data_vars if {"latitude", "longitude"} & set(ds[v].dims)}
    static_vars = [v for v in ds.data_vars if "time" not in ds[v].dims]
    time_idx = pd.DatetimeIndex(ds["time"].values, name="Valid time (UTC)")
    lead = ds["lead_hours"].values if "lead_hours" in ds.coords else None

    # one sheet per time-varying variable: rows = time, columns = sub-basins
    sheets, used = {}, {"Summary", "Variables"}
    for var in time_vars:
        if var in spatial:
            df = pd.DataFrame({n: np.asarray(series[n][var].values) for n in series}, index=time_idx)
        else:                                  # no spatial dimension: one column, not N copies
            df = pd.DataFrame({"value": np.asarray(ds[var].values)}, index=time_idx)
        if lead is not None:
            df.insert(0, "Lead hour", lead)
        sheets[sheet_name(var, used)] = df

    # summary: totals for accumulated variables, means for the rest, plus static fields
    summary = {"Sub-basin": list(series), "Grid cells": [n_cells[n] for n in series]}
    for var in static_vars:
        unit = ds[var].attrs.get("units", "")
        summary[f"{var} [{unit}]"] = [float(series[n][var].values) for n in series]
    for var in time_vars:
        unit = ds[var].attrs.get("units", "")
        vals = {n: np.asarray(series[n][var].values, dtype="float64") for n in series}
        if any(k in var for k in SUM_LIKE):
            summary[f"{var} total [{unit}]"] = [float(np.nansum(vals[n])) for n in series]
            summary[f"{var} peak [{unit}]"] = [float(np.nanmax(vals[n])) for n in series]
        else:
            summary[f"{var} mean [{unit}]"] = [float(np.nanmean(vals[n])) for n in series]
            summary[f"{var} min [{unit}]"] = [float(np.nanmin(vals[n])) for n in series]
            summary[f"{var} max [{unit}]"] = [float(np.nanmax(vals[n])) for n in series]
    summary_df = pd.DataFrame(summary)

    # data dictionary so the sheet names can stay short
    dictionary = pd.DataFrame({
        "Variable": list(ds.data_vars),
        "Units": [ds[v].attrs.get("units", "") for v in ds.data_vars],
        "Description": [ds[v].attrs.get("long_name", "") for v in ds.data_vars],
        "Spatial aggregation": ["cos(latitude)-weighted basin mean" if v in spatial
                                else "none (no spatial dimension)" for v in ds.data_vars],
        "Summary statistic": ["basin value" if v in static_vars
                              else "total and peak" if any(k in v for k in SUM_LIKE)
                              else "mean, min, max" for v in ds.data_vars],
    })

    with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
        summary_df.to_excel(xl, sheet_name="Summary", index=False)
        dictionary.to_excel(xl, sheet_name="Variables", index=False)
        for name, df in sheets.items():
            df.to_excel(xl, sheet_name=name)
    ds.close()

    written.append(xlsx)
    print(f"wrote {xlsx}: Summary + {len(sheets)} variable sheet(s), "
          f"{len(series)} sub-basins x {len(time_idx)} steps")

summary_df

In [ ]:
#@title Optional — plot and download
import matplotlib.pyplot as plt

book = written[0]
precip = pd.read_excel(book, sheet_name=[s for s in pd.ExcelFile(book).sheet_names
                                         if s.startswith("precipitation")][0], index_col=0)
eto = pd.read_excel(book, sheet_name=[s for s in pd.ExcelFile(book).sheet_names
                                      if s.startswith("eto")][0], index_col=0)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
precip.drop(columns="Lead hour", errors="ignore").plot(ax=axes[0], legend=False)
axes[0].set_ylabel("Precipitation"); axes[0].set_title(f"Sub-basin means — {book}")
eto.drop(columns="Lead hour", errors="ignore").plot(ax=axes[1], legend=False)
axes[1].set_ylabel("ET0")
fig.tight_layout()

try:
    from google.colab import files
    for f in written:
        files.download(f)
except ImportError:
    pass